# Análisis de usabilidad — SUS (System Usability Scale) — *pendiente de datos*

Plantilla preparada para el análisis de la encuesta SUS del plan de pruebas
de usuario de la guía. **Hoy no hay datos reales** (`docs/mediciones/sus/`
aún no existe: las pruebas de usuario no se han corrido), así que este
notebook **no fabrica resultados** (D.1 exige evidencia real, no simulada).

Al abrirlo se confirma el estado en la celda de abajo. En cuanto el equipo
de pruebas entregue `docs/mediciones/sus/respuestas-sus.csv` con una fila
por participante (columnas `participante, item_1..item_10`, escala 1-5),
este notebook se ejecuta de punta a punta y sus outputs quedan archivados
como evidencia — mismo patrón que `perf-analysis.ipynb`.


## Reglas de puntuación SUS (Brooke, 1986) — implementadas en la celda de abajo

1. Cada ítem impar (1,3,5,7,9): `valor - 1`.
2. Cada ítem par (2,4,6,8,10): `5 - valor`.
3. Suma de los 10 ajustes × 2.5 → puntuación SUS 0–100 por participante.
4. Interpretación estándar (Bangor et al.): ≥ 70 aceptable; 50–70 marginal;
   < 50 no aceptable.

Nota de reproducibilidad (D.2): este análisis es agregación pura sobre
datos ya recolectados, **no usa muestreo aleatorio** — no requiere semilla
(caso "no aplica, confirmado" documentado en el commit de esta rama).


In [1]:
import os
import subprocess
import sys
from pathlib import Path

def raiz_repo():
    d = Path(os.path.abspath("")).resolve()
    while not (d / "Makefile").is_file() and d != d.parent:
        d = d.parent
    return d

REPO = raiz_repo()
SUS_CSV = REPO / "docs/mediciones/sus/respuestas-sus.csv"

if not SUS_CSV.exists():
    print("ESTADO: aun no hay datos SUS (falta docs/mediciones/sus/respuestas-sus.csv).")
    print("Las celdas de analisis estan listas abajo; se ejecutaran en cuanto")
    print("existan respuestas reales de las pruebas de usuario.")
else:
    print("Datos SUS encontrados, ejecutando analisis real.")


ESTADO: aun no hay datos SUS (falta docs/mediciones/sus/respuestas-sus.csv).
Las celdas de analisis estan listas abajo; se ejecutaran en cuanto
existan respuestas reales de las pruebas de usuario.


In [2]:
import csv
import statistics

def puntuar_sus(fila):
    """Puntuacion SUS de una fila (Brooke 1986): items impares valor-1,
    items pares 5-valor, suma*2.5."""
    items = [int(fila[f"item_{i}"]) for i in range(1, 11)]
    ajustados = [(v - 1) if i % 2 == 1 else (5 - v) for i, v in enumerate(items, start=1)]
    return sum(ajustados) * 2.5

def analizar_sus(path):
    with open(path, newline="", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))
    puntajes = [puntuar_sus(f) for f in filas]
    n = len(puntajes)
    media = statistics.mean(puntajes)
    desv = statistics.stdev(puntajes) if n > 1 else float("nan")
    print(f"Participantes: {n}")
    print(f"Puntuacion SUS por participante: {[round(p, 1) for p in puntajes]}")
    print(f"Media: {media:.1f} | Desviacion tipica: {desv:.1f} | Min: {min(puntajes):.1f} | Max: {max(puntajes):.1f}")
    if media >= 70:
        print("Interpretacion: aceptable (>= 70)")
    elif media >= 50:
        print("Interpretacion: marginal (50-70)")
    else:
        print("Interpretacion: no aceptable (< 50)")
    return puntajes

if SUS_CSV.exists():
    analizar_sus(SUS_CSV)
